# JPmart - Gold Aggregation

Builds a star schema from Silver (dimensions + fact tables), plus a set
of business-ready aggregate tables computed on top of that star schema.

**Design notes:**
- Dimensions use a Type 1 (overwrite) pattern — Silver itself is already
  a full overwrite of current state, so there's no change history in
  Gold to preserve either. A production version tracking how a
  customer's attributes changed over time would need Type 2 instead.
- `dim_customers` and `dim_products` each include a synthetic `UNKNOWN`
  member row, so fact tables never carry a null foreign key for a
  legitimately-missing reference (e.g. an anonymous web session) —
  a standard dimensional-modeling technique.

## Configuration

In [0]:
from pyspark.sql.functions import (
    col, lit, when, coalesce,
    sum as spark_sum, count, countDistinct, avg,
    min as spark_min, max as spark_max, round as spark_round,
    date_format, year, quarter, month, dayofmonth, dayofweek,
    to_date, sequence, explode, percent_rank, lag,
)
from pyspark.sql.window import Window
from datetime import date

CATALOG = "jpmart"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

UNKNOWN_KEY = "UNKNOWN"

# Web event funnel stage order -- used by fact_web_events and
# agg_conversion_funnel to know how far a session progressed.
FUNNEL_STAGES = {
    "page_view": 1,
    "product_view": 2,
    "add_to_cart": 3,
    "checkout_start": 4,
    "purchase": 5,
}

## Shared helper — write Gold table

In [0]:
def write_gold_table(df, table_name):
    """Writes a Gold dimension, fact, or aggregate table as a full
    overwrite. Gold is rebuilt from Silver each run rather than
    maintained incrementally -- appropriate at this data volume, and
    keeps the aggregation logic simple to read and verify."""
    target_table = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"  -> {target_table}: {df.count():,} rows")

## Dimension tables

In [0]:
def build_dim_date():
    """Generates a standalone calendar dimension covering every date
    that can appear in any fact table (order/signup/event dates), rather
    than deriving it from the fact tables themselves -- a date dimension
    should exist independently and carry full calendar context (quarter,
    day-of-week name) that isn't naturally present in the source data.
    """
    start_date = "2023-01-01"  # covers the -3y signup / -2y order window with margin
    end_date = date.today().isoformat()

    dim = (
        spark.createDataFrame([(1,)], ["_seed"])
        .select(explode(sequence(to_date(lit(start_date)), to_date(lit(end_date)))).alias("date"))
        .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
        .withColumn("year", year(col("date")))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("month_name", date_format(col("date"), "MMMM"))
        .withColumn("day", dayofmonth(col("date")))
        .withColumn("day_name", date_format(col("date"), "EEEE"))
        .withColumn("is_weekend", dayofweek(col("date")).isin(1, 7))  # Spark: 1=Sun, 7=Sat
        .select("date_key", "date", "year", "quarter", "month", "month_name", "day", "day_name", "is_weekend")
    )
    return dim

In [0]:
def build_dim_customers():
    """Builds dim_customers from Silver, plus one synthetic UNKNOWN
    member row. web_events has legitimate anonymous sessions (null
    customer_id) -- rather than leaving a null FK in fact_web_events,
    every anonymous event points at this row instead.
    """
    cols = ["customer_id", "first_name", "last_name", "email", "state", "loyalty_member", "signup_date"]
    silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.customers").select(*cols)

    unknown_row = spark.createDataFrame(
        [(UNKNOWN_KEY, "Anonymous", "Session", None, None, None, None)],
        schema=silver.schema,
    )

    return silver.unionByName(unknown_row)

In [0]:
def build_dim_products():
    """Same UNKNOWN-member pattern as dim_customers: some web_events have
    no associated product (e.g. a page_view on the homepage), so
    fact_web_events points those at this row instead of a null FK.
    """
    cols = ["product_id", "product_name", "category", "supplier", "unit_price", "active"]
    silver = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.products").select(*cols)

    unknown_row = spark.createDataFrame(
        [(UNKNOWN_KEY, "N/A", "N/A", None, None, None)],
        schema=silver.schema,
    )

    return silver.unionByName(unknown_row)

## Fact tables

In [0]:
def build_fact_orders():
    """Grain: one row per order_id. total_amount/total_units are derived
    from Silver order_items rather than stored redundantly upstream --
    orders and order_items are independent Silver tables.

    Known limitation: an order_item with an unresolved quantity or
    unit_price (nulled during Silver cleaning, e.g. a negative capture
    error) doesn't contribute to the sum -- Spark's sum() skips nulls.
    This can slightly undercount total_amount for an affected order,
    called out here deliberately rather than hidden.

    Orders quarantined in Silver (missing customer_id) are already
    excluded upstream, so every row here has a real customer FK -- no
    UNKNOWN member needed, unlike fact_web_events.
    """
    orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.orders")
    items = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.order_items")

    order_totals = (
        items
        .withColumn("line_total", col("quantity") * col("unit_price"))
        .groupBy("order_id")
        .agg(
            spark_sum("line_total").alias("total_amount"),
            spark_sum("quantity").alias("total_units"),
        )
    )

    fact = (
        orders
        .join(order_totals, on="order_id", how="left")
        .withColumn("date_key", date_format(col("order_date"), "yyyyMMdd").cast("int"))
        .select("order_id", "customer_id", "date_key", "order_date", "status", "total_amount", "total_units")
    )
    return fact

In [0]:
def build_fact_order_items():
    """Grain: one row per order_item_id (the transaction-line grain).
    product_id is guaranteed present -- rows missing it were already
    quarantined in Silver -- so, unlike fact_web_events, there's no need
    for an UNKNOWN dim_products member here.

    line_total is null wherever quantity or unit_price is null (an
    unresolved capture error). The row still exists -- it's a valid
    order_item, just with an unknown monetary value -- but any revenue
    aggregate below filters these out explicitly.
    """
    items = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.order_items")
    return items.withColumn("line_total", col("quantity") * col("unit_price"))

In [0]:
def build_fact_web_events():
    """Grain: one row per event_id. customer_id/product_id are coalesced
    to the UNKNOWN dim member instead of left null -- an anonymous
    session or a product-less page_view is a normal, expected state in
    clickstream data, not a data-quality problem, so it gets a real
    dimension row to join against.

    stage_order encodes each event's position in the funnel
    (page_view=1 ... purchase=5), consumed by agg_conversion_funnel below.
    """
    events = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.web_events")

    stage_map = (
        when(col("event_type") == "page_view", 1)
        .when(col("event_type") == "product_view", 2)
        .when(col("event_type") == "add_to_cart", 3)
        .when(col("event_type") == "checkout_start", 4)
        .when(col("event_type") == "purchase", 5)
    )

    fact = (
        events
        .withColumn("customer_id", coalesce(col("customer_id"), lit(UNKNOWN_KEY)))
        .withColumn("product_id", coalesce(col("product_id"), lit(UNKNOWN_KEY)))
        .withColumn("date_key", date_format(col("event_timestamp"), "yyyyMMdd").cast("int"))
        .withColumn("stage_order", stage_map)
    )
    return fact

## Business aggregates

Computed on top of the Gold star schema itself, not directly from
Silver -- demonstrating that the dimensional model is actually queryable
end to end, not just structurally present.

In [0]:
def build_agg_revenue_by_month_category():
    """Revenue & sales: monthly revenue and units sold per category."""
    items = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
    orders = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    products = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_products")
    dates = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")

    return (
        items
        .filter(col("line_total").isNotNull())
        .join(orders.select("order_id", "date_key"), on="order_id", how="inner")
        .join(products.select("product_id", "category"), on="product_id", how="left")
        .join(dates.select("date_key", "year", "month", "month_name"), on="date_key", how="left")
        .groupBy("year", "month", "month_name", "category")
        .agg(
            spark_round(spark_sum("line_total"), 2).alias("revenue"),
            spark_sum("quantity").alias("units_sold"),
            countDistinct("order_id").alias("order_count"),
        )
        .orderBy("year", "month", "category")
    )

In [0]:
def build_agg_customer_ltv():
    """Customer analytics: lifetime spend per customer, plus a simple
    spend-based tier (High/Medium/Low, by percentile rank). This is a
    simplified stand-in for full RFM scoring -- it only captures the
    Monetary dimension -- labeled as such rather than as a complete
    RFM model.

    Note: Window.orderBy with no partitionBy pulls all rows into a
    single partition to rank them -- fine at this volume (~1,000
    customers), but would need a different approach at real e-commerce
    scale.
    """
    orders = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
    customers = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customers")

    per_customer = (
        orders
        .filter(col("customer_id") != UNKNOWN_KEY)  # defensive; fact_orders should never actually have this
        .groupBy("customer_id")
        .agg(
            spark_round(spark_sum("total_amount"), 2).alias("total_spent"),
            count("order_id").alias("order_count"),
            spark_round(avg("total_amount"), 2).alias("avg_order_value"),
            spark_min("order_date").alias("first_order_date"),
            spark_max("order_date").alias("last_order_date"),
        )
    )

    tier_window = Window.orderBy(col("total_spent"))

    return (
        per_customer
        .join(customers.select("customer_id", "first_name", "last_name", "state"), on="customer_id", how="left")
        .withColumn("spend_percentile", percent_rank().over(tier_window))
        .withColumn(
            "spend_tier",
            when(col("spend_percentile") >= 0.8, "High")
            .when(col("spend_percentile") >= 0.4, "Medium")
            .otherwise("Low"),
        )
        .drop("spend_percentile")
    )

In [0]:
def build_agg_top_products():
    """Product performance: units sold, revenue, and realized average
    selling price per product. avg_selling_price is revenue / units_sold
    (the actual realized average), not a static read of
    dim_products.unit_price -- the two can differ since unit_price on an
    order_item is captured at time of sale.
    """
    items = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
    products = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_products")

    return (
        items
        .filter(col("line_total").isNotNull())
        .groupBy("product_id")
        .agg(
            spark_sum("quantity").alias("units_sold"),
            spark_round(spark_sum("line_total"), 2).alias("revenue"),
            countDistinct("order_id").alias("order_count"),
        )
        .withColumn("avg_selling_price", spark_round(col("revenue") / col("units_sold"), 2))
        .join(products.select("product_id", "product_name", "category"), on="product_id", how="left")
        .orderBy(col("revenue").desc())
    )

In [0]:
def build_agg_conversion_funnel():
    """Web funnel & conversion: per month, how many distinct sessions
    reached each funnel stage, and the stage-over-stage conversion rate.

    Intermediate grain: one row per session_id -- its furthest stage
    reached (max(stage_order)) and the month of its first event. A
    session is then counted toward every stage up to and including that
    furthest one (cumulative funnel reading, e.g. a session that reached
    add_to_cart counts in page_view, product_view, AND add_to_cart).
    """
    events = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_web_events")
    dates = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_date")

    session_progress = (
        events
        .filter(col("stage_order").isNotNull())
        .groupBy("session_id")
        .agg(
            spark_max("stage_order").alias("max_stage"),
            spark_min("date_key").alias("date_key"),
        )
        .join(dates.select("date_key", "year", "month", "month_name"), on="date_key", how="left")
    )

    stage_dfs = [
        session_progress
        .filter(col("max_stage") >= stage_num)
        .groupBy("year", "month", "month_name")
        .agg(count("session_id").alias("session_count"))
        .withColumn("stage", lit(stage_name))
        .withColumn("stage_order", lit(stage_num))
        for stage_name, stage_num in FUNNEL_STAGES.items()
    ]

    funnel = stage_dfs[0]
    for df in stage_dfs[1:]:
        funnel = funnel.unionByName(df)

    stage_window = Window.partitionBy("year", "month").orderBy("stage_order")

    return (
        funnel
        .withColumn("previous_stage_count", lag("session_count").over(stage_window))
        .withColumn(
            "conversion_rate",
            when(col("previous_stage_count").isNotNull(),
                 spark_round(col("session_count") / col("previous_stage_count"), 4)),
        )
        .drop("previous_stage_count")
        .orderBy("year", "month", "stage_order")
    )

## Build Gold — dimensions, then facts, then aggregates

Aggregates query the Gold fact/dim tables themselves (not Silver), so
they have to run after both are written.

In [0]:
DIMENSIONS = [("dim_date", build_dim_date), ("dim_customers", build_dim_customers), ("dim_products", build_dim_products)]
FACTS = [("fact_orders", build_fact_orders), ("fact_order_items", build_fact_order_items), ("fact_web_events", build_fact_web_events)]
AGGREGATES = [
    ("agg_revenue_by_month_category", build_agg_revenue_by_month_category),
    ("agg_customer_ltv", build_agg_customer_ltv),
    ("agg_top_products", build_agg_top_products),
    ("agg_conversion_funnel", build_agg_conversion_funnel),
]

for name, build_fn in DIMENSIONS + FACTS + AGGREGATES:
    write_gold_table(build_fn(), name)

## Validation

Referential integrity: every FK in a fact table should resolve to a
real row in its dimension (including the UNKNOWN member where used).

In [0]:
fact_orders = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_orders")
fact_order_items = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_order_items")
fact_web_events = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.fact_web_events")
dim_customers = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_customers")
dim_products = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.dim_products")

checks = [
    ("fact_orders -> dim_customers", fact_orders.select("customer_id"), dim_customers.select("customer_id")),
    ("fact_order_items -> dim_products", fact_order_items.select("product_id"), dim_products.select("product_id")),
    ("fact_order_items -> fact_orders", fact_order_items.select("order_id"), fact_orders.select("order_id")),
    ("fact_web_events -> dim_customers", fact_web_events.select("customer_id"), dim_customers.select("customer_id")),
    ("fact_web_events -> dim_products", fact_web_events.select("product_id"), dim_products.select("product_id")),
]

for label, child, parent in checks:
    orphans = child.distinct().join(parent.distinct(), on=child.columns[0], how="left_anti").count()
    print(f"{label} orphans: {orphans} (expected: 0)")

display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.agg_revenue_by_month_category").limit(20))
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.agg_customer_ltv").orderBy(col("total_spent").desc()).limit(10))
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.agg_top_products").limit(10))
display(spark.table(f"{CATALOG}.{GOLD_SCHEMA}.agg_conversion_funnel").limit(20))